## Credit Risk Modeling — Feature Engineering

Purpose of This Notebook

This notebook transforms cleaned transactional and behavioral data into model-ready risk features by:

- Aggregating repayment behavior over time

- Capturing delinquency severity and trends

- Quantifying utilization and exposure dynamics

- Engineering stability vs volatility signals

No modeling is performed in this notebook.
The goal is to convert raw history into predictive risk signals.

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np

### Load the Cleaned Dataset

In [ ]:
df= pd.read_excel("default of credit card clients.xls", header=1)
df=df.drop(columns=["ID"])
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Define Feature Groups

In [ ]:
repayment_cols=['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
bill_cols=['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
payment_cols=['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

### Delinquency Severity Features

### Avg Delinquency

In [ ]:
df['Average_Delinquency']= df[repayment_cols].mean(axis=1)

### Maximum Delinquency

In [ ]:
df['Max_Delinquency']= df[repayment_cols].max(axis=1)

### Number of Delinquency months

In [ ]:
df['number_of_delinquency_months']= (df[repayment_cols]>0).sum(axis=1)

### Delinquency Severity Features

These features summarize the intensity and persistence of historical repayment delays.

- **Average_Delinquency** captures the overall repayment behavior across six months.
- **Max_Delinquency** reflects the worst observed delinquency event, which is critical for tail-risk identification.
- **number_of_delinquency_months** measures how frequently the customer was delinquent.

From the distribution:
- Most customers have low or zero delinquency on average.
- A smaller but significant subset exhibits repeated or severe delinquency.
- These features strongly differentiate stable payers from chronically risky borrowers.

In credit risk modeling, delinquency history is typically the strongest predictor of future default.


### Delinquency Trend Feature

In [ ]:
df['delinquency_trend']= df['PAY_0']- df['PAY_6']
df['delinquency_trend']

,delinquency_trend
0,4
1,-3
2,0
3,0
4,-1
...,...
29995,0
29996,-1
29997,4
29998,2


### Delinquency Trend Interpretation

The delinquency trend measures changes in repayment behavior over time by comparing the most recent month to the oldest month.

- **Positive values** indicate worsening repayment behavior.
- **Zero** indicates stable behavior.
- **Negative values** suggest improving repayment discipline.

The observed distribution shows that:
- Most customers exhibit stable repayment patterns.
- A meaningful portion shows worsening trends, which is a leading indicator of default risk.
- Trend-based features add forward-looking signal beyond static delinquency measures.


### Credit utilization features

### Average Utilization

In [ ]:
df['avg_utilization']= (df[bill_cols].mean(axis=1)/df['LIMIT_BAL']).replace([np.inf, -np.inf],0).fillna(0)

### Maximum Utilization

In [ ]:
df['max_utilization']= (df[bill_cols].max(axis=1)/df['LIMIT_BAL']).replace([np.inf, -np.inf], 0).fillna(0)

### Credit Utilization Features

Credit utilization measures how much of the approved credit limit is being used.

- **avg_utilization** reflects typical borrowing behavior.
- **max_utilization** captures peak stress periods.

Key observations:
- Median utilization is moderate, but the upper tail is extremely high.
- Some utilization values exceed 1 due to negative balances or timing effects in billing data.
- High utilization is a well-established early warning signal for financial stress and default.

These features proxy liquidity pressure and exposure intensity.


### Payment Behavior Ratios

### Total Payments VS Total Bills

In [ ]:
df['payment_to_bill_ratio']= (df[payment_cols].sum(axis=1)/ df[bill_cols].sum(axis=1)).replace([np.inf, -np.inf], 0).fillna(0)

### Payment-to-Bill Ratio

This ratio compares total payments made to total billed amounts.

Interpretation:
- Values near 1 indicate full or near-full repayment.
- Low values indicate revolving balances or partial payments.
- Extremely high or negative values arise from billing credits or adjustments.

Despite noise in extreme values, this feature captures repayment discipline and complements delinquency-based indicators.


### Payment Stability Ratios

### Payment Volatility

In [ ]:
df['payment_std']= df[payment_cols].std(axis=1)

### Average Monthly Payment

In [ ]:
df['Avg_payment']= df[payment_cols].mean(axis=1)

### Payment Stability Metrics

- **payment_std** measures volatility in monthly payments.
- **Avg_payment** captures typical repayment capacity.

Insights:
- High volatility suggests unstable income or liquidity stress.
- Stable, consistent payments are associated with lower default risk.
- Payment variability provides behavioral context not captured by totals alone.


### Exposure Dynamics

### Bill Volatility

In [ ]:
df['bill_std']= df[bill_cols].std(axis=1)

Average Monthly Balance

In [ ]:
df['Avg_bill']= df[bill_cols].mean(axis=1)

### Exposure Dynamics

- **bill_std** captures volatility in outstanding balances.
- **Avg_bill** reflects typical exposure level.

Observations:
- Large balance volatility indicates fluctuating credit usage.
- Higher average balances increase loss severity in default scenarios.
- These features help distinguish high-risk, high-exposure customers from low-impact defaulters.


### Risk Flag Indicators

In [ ]:
df['high_utilization_flag']= (df['max_utilization']>0.8).astype(int)
df['chronic_delinquency_flag']= (df['number_of_delinquency_months']>=3).astype(int)
df['severe_delinquency_flag']= (df['Max_Delinquency']>=3).astype(int)

### Risk Flag Indicators

Binary flags were created to capture key high-risk patterns:

- **high_utilization_flag** identifies customers frequently near their credit limit.
- **chronic_delinquency_flag** captures persistent delinquency behavior.
- **severe_delinquency_flag** isolates customers with serious repayment failures.

These flags:
- Improve interpretability
- Support rule-based decisioning
- Are especially useful for tree-based models and policy frameworks


### Feature Set Review

In [ ]:
engineered_features=['Average_Delinquency', 'Max_Delinquency', 'number_of_delinquency_months', 'delinquency_trend', 'avg_utilization',
                     'max_utilization', 'payment_to_bill_ratio', 'payment_std', 'Avg_payment', 'bill_std', 'Avg_bill',
                     'high_utilization_flag', 'chronic_delinquency_flag', 'severe_delinquency_flag']
df[engineered_features].describe()

,Average_Delinquency,Max_Delinquency,number_of_delinquency_months,delinquency_trend,avg_utilization,max_utilization,payment_to_bill_ratio,payment_std,Avg_payment,bill_std,Avg_bill,high_utilization_flag,chronic_delinquency_flag,severe_delinquency_flag
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,-0.182439,0.438733,0.834200,0.274400,0.373048,0.494965,0.380941,5804.918519,5275.232094,12077.787549,44976.945200,0.319333,0.124800,0.039767
std,0.982176,1.345154,1.554303,1.165683,0.351890,0.433046,7.671004,15006.438973,10137.946323,20302.149149,63260.721860,0.466226,0.330498,0.195414
min,-2.000000,-2.000000,0.000000,-7.000000,-0.232590,-0.100000,-546.928571,0.000000,0.000000,0.000000,-56043.166667,0.000000,0.000000,0.000000
25%,-0.833333,0.000000,0.000000,0.000000,0.029997,0.070636,0.040952,615.353672,1113.291667,1549.935213,4781.333333,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.284834,0.430702,0.084932,1413.694125,2397.166667,4579.662158,21051.833333,0.000000,0.000000,0.000000
75%,0.000000,2.000000,1.000000,1.000000,0.687929,0.923246,0.586922,4126.957632,5583.916667,14352.259579,57104.416667,1.000000,0.000000,0.000000
max,6.000000,8.000000,6.000000,7.000000,5.364308,10.688575,797.000000,650098.338820,627344.333333,647788.051081,877313.833333,1.000000,1.000000,1.000000


### Feature Engineering Summary

This notebook transformed raw transactional credit data into structured, model-ready risk indicators.

Key enhancements include:
- Delinquency severity, frequency, and trend metrics
- Credit utilization and exposure ratios
- Payment discipline and volatility indicators
- Binary flags for severe and chronic risk patterns

The resulting dataset expands from 24 raw variables to 40 analytically meaningful features while preserving business interpretability.

These engineered features encode behavioral risk dynamics that are not directly observable in raw data and form a strong foundation for predictive modeling.

The dataset is now ready for supervised learning, risk segmentation, and loss modeling.


In [ ]:
df.shape

(30000, 38)

In [ ]:
df.to_excel('default of credit card clients_final.xlsx', header=True, index=False)